In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
from IPython.core.display import HTML
import matplotlib.pyplot as plt
from scipy.stats import uniform

In [3]:
data = pd.read_csv("heart.csv")
data.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [3]:
data.tail()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
298,57,0,0,140,241,0,1,123,1,0.2,1,0,3,0
299,45,1,3,110,264,0,1,132,0,1.2,1,0,3,0
300,68,1,0,144,193,1,1,141,0,3.4,1,2,3,0
301,57,1,0,130,131,0,1,115,1,1.2,1,1,3,0
302,57,0,1,130,236,0,0,174,0,0.0,1,1,2,0


In [4]:
data.shape

(303, 14)

In [5]:
thal = data['thal']
thal

0      1
1      2
2      2
3      2
4      2
      ..
298    3
299    3
300    3
301    3
302    2
Name: thal, Length: 303, dtype: int64

In [6]:
data.dtypes

age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64
dtype: object

In [7]:
data.info

<bound method DataFrame.info of      age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
0     63    1   3       145   233    1        0      150      0      2.3   
1     37    1   2       130   250    0        1      187      0      3.5   
2     41    0   1       130   204    0        0      172      0      1.4   
3     56    1   1       120   236    0        1      178      0      0.8   
4     57    0   0       120   354    0        1      163      1      0.6   
..   ...  ...  ..       ...   ...  ...      ...      ...    ...      ...   
298   57    0   0       140   241    0        1      123      1      0.2   
299   45    1   3       110   264    0        1      132      0      1.2   
300   68    1   0       144   193    1        1      141      0      3.4   
301   57    1   0       130   131    0        1      115      1      1.2   
302   57    0   1       130   236    0        0      174      0      0.0   

     slope  ca  thal  target  
0        0   0     1    

In [8]:
# This line filters the dataset called "data" and keeps only the rows
# where the value in the column named 'ca' is less than 4.

# Here's what happens step-by-step:
# 1. data['ca']   → selects the column 'ca' from the DataFrame (like one column in Excel).
# 2. data['ca'] < 4 → compares each value in that column to 4, giving True or False for every row.
#                     Example: [0, 2, 4, 3] becomes [True, True, False, True].
# 3. data[data['ca'] < 4] → uses those True/False results to keep only the rows marked True.
#                           So rows where 'ca' >= 4 are removed.
# 4. The "=" sign assigns this new filtered DataFrame back to the same variable name "data",
#    replacing the old one with the cleaned version.
#
# In short: this line removes rows where 'ca' has an invalid or impossible value (4 or more).
data = data[data['ca'] < 4]  # drop rows with invalid 'ca' values


# This line works the same way as the one above, but for the column 'thal'.
#
# Step-by-step:
# 1. data['thal'] → selects the column named 'thal' from the dataset.
# 2. data['thal'] > 0 → checks each value to see if it’s greater than zero,
#                       producing a list of True/False values for every row.
# 3. data[data['thal'] > 0] → keeps only the rows where that condition is True.
#                             Any row with 'thal' <= 0 is dropped.
# 4. Again, the "=" sign replaces the old dataset with the new filtered one.
#
# In short: this removes rows with missing or invalid 'thal' values (0 or negative).
data = data[data['thal'] > 0]  # drop rows with invalid 'thal' values


# This line prints out how many rows (entries) remain in the dataset after cleaning.
#
# Step-by-step:
# 1. len(data) → counts the number of rows currently in the DataFrame.
# 2. The 'f' before the string means "formatted string" (f-string). It lets you insert
#    variables directly into text using curly braces {}.
# 3. {len(data)} is replaced by the actual number of rows when printed.
# 4. The text inside print() is then shown in the output window or console.
#
# Example output: "The length of the data now is 299 instead of 303!"
print(f'The length of the data now is {len(data)} instead of 303!')

The length of the data now is 296 instead of 303!


In [9]:
# This line renames several column names in the DataFrame called "data"
# to make them more descriptive and easier to understand.
#
# In pandas, a DataFrame is like a spreadsheet with rows and columns.
# The method .rename() is used to change the names of one or more columns or rows.
#
# Syntax breakdown:
# data.rename(columns = {...}, errors = "raise")
# └── 'columns' tells pandas which columns to rename and what their new names should be.
#     You provide these as a dictionary: {'old_name': 'new_name', 'old_name2': 'new_name2', ...}
# └── 'errors' controls what happens if you try to rename a column that doesn’t exist.
#     - "raise" means: if any of the old column names are missing, stop the code and show an error.
#     - "ignore" would mean: skip missing ones and rename only those that exist.
#
# Let’s go through what’s inside the dictionary step by step:
# Each item inside { } has a key (before the :) and a value (after the :).
# The key is the old column name, and the value is the new column name.
#
# For example:
#   'cp': 'chest_pain_type'      → Rename the column 'cp' to 'chest_pain_type'
#   'trestbps': 'resting_blood_pressure' → Rename 'trestbps' to 'resting_blood_pressure'
#   'chol': 'cholesterol'        → Rename 'chol' to 'cholesterol'
#   'fbs': 'fasting_blood_sugar' → Rename 'fbs' to 'fasting_blood_sugar'
#   'restecg': 'resting_electrocardiogram' → Rename 'restecg' to 'resting_electrocardiogram'
#   'thalach': 'max_heart_rate_achieved'  → Rename 'thalach' to 'max_heart_rate_achieved'
#   'exang': 'exercise_induced_angina'    → Rename 'exang' to 'exercise_induced_angina'
#   'oldpeak': 'st_depression'   → Rename 'oldpeak' to 'st_depression'
#   'slope': 'st_slope'          → Rename 'slope' to 'st_slope'
#   'ca': 'num_major_vessels'    → Rename 'ca' to 'num_major_vessels'
#   'thal': 'thalassemia'        → Rename 'thal' to 'thalassemia'
#
# The outer parentheses are used to make the code block visually clearer and prevent errors
# when the dictionary spans multiple lines.
#
# Finally, the entire operation assigns the result back to 'data'.
# That means the DataFrame "data" is updated with these new column names.
data = data.rename(
    columns = {
        'cp': 'chest_pain_type', 
        'trestbps': 'resting_blood_pressure', 
        'chol': 'cholesterol',
        'fbs': 'fasting_blood_sugar',
        'restecg': 'resting_electrocardiogram', 
        'thalach': 'max_heart_rate_achieved', 
        'exang': 'exercise_induced_angina',
        'oldpeak': 'st_depression', 
        'slope': 'st_slope', 
        'ca': 'num_major_vessels', 
        'thal': 'thalassemia'
    }, 
    errors = "raise"  # If any old column name doesn’t exist, show an error instead of ignoring it
)

In [10]:
data.head()

,age,sex,chest_pain_type,resting_blood_pressure,cholesterol,fasting_blood_sugar,resting_electrocardiogram,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [16]:
# The following lines safely replace numeric category codes in the dataset
# with meaningful text labels using .loc[] — a pandas method that allows you
# to select specific rows and columns at the same time, without warnings or ambiguity.
#
# This process is called data labeling or value mapping.
# It helps make your dataset easier to read, understand, and interpret for humans.

# --------------------------------------------------------------------
# SYNTAX REMINDER
# --------------------------------------------------------------------
# The general structure of each line looks like this:
#
#   data.loc[row_condition, 'column_name'] = new_value
#
# Let’s break it down clearly:
# 1. data → your dataset, stored as a "DataFrame" (like an Excel table with rows and columns).
# 2. .loc[] → means "locate". It’s a pandas function used to find and modify specific rows
#    and columns that meet certain conditions.
# 3. Inside the square brackets [ ], there are two parts separated by a comma:
#       - The first part (before the comma) specifies which rows to target.
#         Example: data['sex'] == 0 → find rows where the value in the 'sex' column equals 0.
#       - The second part (after the comma) specifies which column to modify.
#         Example: 'sex' → modify the column named 'sex'.
# 4. == → means "is equal to". It checks whether each value matches the number you specify.
#    Example: [1, 0, 1] == 0 → [False, True, False]
# 5. = (a single equal sign) → means "assign" or "replace". It changes the old value
#    with a new one in the selected cells.
# 6. The text in quotes (like 'female') → is a string, meaning plain text.
#    Strings are used to store words or labels, not numbers.

# --------------------------------------------------------------------
# SEX COLUMN
# --------------------------------------------------------------------
# The 'sex' column uses numbers to represent gender:
# 0 = female, 1 = male.
# These lines replace those numbers with the actual text labels.

# If the value is 0, replace it with 'female'
data.loc[data['sex'] == 0, 'sex'] = 'female'

# If the value is 1, replace it with 'male'
data.loc[data['sex'] == 1, 'sex'] = 'male'


# --------------------------------------------------------------------
# CHEST PAIN TYPE COLUMN
# --------------------------------------------------------------------
# The 'chest_pain_type' column contains four numeric categories representing 
# different types of chest pain. These are replaced with descriptive text.

# 0 = typical angina (chest pain caused by reduced blood flow)
data.loc[data['chest_pain_type'] == 0, 'chest_pain_type'] = 'typical angina'

# 1 = atypical angina (chest pain not related to physical exertion)
data.loc[data['chest_pain_type'] == 1, 'chest_pain_type'] = 'atypical angina'

# 2 = non-anginal pain (chest pain unrelated to heart problems)
data.loc[data['chest_pain_type'] == 2, 'chest_pain_type'] = 'non-anginal pain'

# 3 = asymptomatic (no chest pain or visible symptoms)
data.loc[data['chest_pain_type'] == 3, 'chest_pain_type'] = 'asymptomatic'


# --------------------------------------------------------------------
# FASTING BLOOD SUGAR COLUMN
# --------------------------------------------------------------------
# This column shows whether a person's fasting blood sugar is above or below 120 mg/dL.
# (Note: the correct unit is mg/dL, not mg/ml.)

# 0 = blood sugar ≤ 120 mg/dL
data.loc[data['fasting_blood_sugar'] == 0, 'fasting_blood_sugar'] = '≤ 120 mg/dL'

# 1 = blood sugar > 120 mg/dL
data.loc[data['fasting_blood_sugar'] == 1, 'fasting_blood_sugar'] = '> 120 mg/dL'


# --------------------------------------------------------------------
# RESTING ELECTROCARDIOGRAM COLUMN
# --------------------------------------------------------------------
# This column records the results of an ECG (electrocardiogram) test.

# 0 = normal
data.loc[data['resting_electrocardiogram'] == 0, 'resting_electrocardiogram'] = 'normal'

# 1 = ST-T wave abnormality (abnormal heart electrical pattern)
data.loc[data['resting_electrocardiogram'] == 1, 'resting_electrocardiogram'] = 'ST-T wave abnormality'

# 2 = left ventricular hypertrophy (thickening of the left heart chamber)
data.loc[data['resting_electrocardiogram'] == 2, 'resting_electrocardiogram'] = 'left ventricular hypertrophy'


# --------------------------------------------------------------------
# EXERCISE-INDUCED ANGINA COLUMN
# --------------------------------------------------------------------
# This column shows whether the patient experienced angina (chest pain)
# during physical exercise.

# 0 = no angina during exercise
data.loc[data['exercise_induced_angina'] == 0, 'exercise_induced_angina'] = 'no'

# 1 = angina occurred during exercise
data.loc[data['exercise_induced_angina'] == 1, 'exercise_induced_angina'] = 'yes'


# --------------------------------------------------------------------
# ST SLOPE COLUMN
# --------------------------------------------------------------------
# The 'st_slope' column describes the slope of the ST segment in an ECG graph.
# This can indicate different heart conditions.

# 0 = upsloping (ST segment rises)
data.loc[data['st_slope'] == 0, 'st_slope'] = 'upsloping'

# 1 = flat (ST segment stays level)
data.loc[data['st_slope'] == 1, 'st_slope'] = 'flat'

# 2 = downsloping (ST segment falls)
data.loc[data['st_slope'] == 2, 'st_slope'] = 'downsloping'


# --------------------------------------------------------------------
# THALASSEMIA COLUMN
# --------------------------------------------------------------------
# Thalassemia is a blood disorder measured here as heart-related defects 
# detected through imaging tests.
# Note: In some datasets, these codes might differ, so always check your unique values first.

# 1 = fixed defect (heart issue that does not change with exercise)
data.loc[data['thalassemia'] == 1, 'thalassemia'] = 'fixed defect'

# 2 = normal (no detected abnormality)
data.loc[data['thalassemia'] == 2, 'thalassemia'] = 'normal'

# 3 = reversible defect (heart issue that improves or disappears during exercise)
data.loc[data['thalassemia'] == 3, 'thalassemia'] = 'reversible defect'


# --------------------------------------------------------------------
# SUMMARY
# --------------------------------------------------------------------
# - .loc[] safely finds and edits rows in a DataFrame without warnings.
# - Each line searches for a numeric code in one column and replaces it with clear text.
# - This process improves readability and prepares the data for visualization or analysis.

C:\Users\Acity\AppData\Local\Temp\ipykernel_8484\1493366153.py:39: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'female' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[data['sex'] == 0, 'sex'] = 'female'
C:\Users\Acity\AppData\Local\Temp\ipykernel_8484\1493366153.py:52: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'typical angina' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[data['chest_pain_type'] == 0, 'chest_pain_type'] = 'typical angina'
C:\Users\Acity\AppData\Local\Temp\ipykernel_8484\1493366153.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '≤ 120 mg/dL' has dtype incompatible with int64, please explicitly cast to a compatible d

In [11]:
data.head()

,age,sex,chest_pain_type,resting_blood_pressure,cholesterol,fasting_blood_sugar,resting_electrocardiogram,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [12]:
data.tail()

,age,sex,chest_pain_type,resting_blood_pressure,cholesterol,fasting_blood_sugar,resting_electrocardiogram,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,target
298,57,0,0,140,241,0,1,123,1,0.2,1,0,3,0
299,45,1,3,110,264,0,1,132,0,1.2,1,0,3,0
300,68,1,0,144,193,1,1,141,0,3.4,1,2,3,0
301,57,1,0,130,131,0,1,115,1,1.2,1,1,3,0
302,57,0,1,130,236,0,0,174,0,0.0,1,1,2,0


In [13]:
data.dtypes

age                            int64
sex                            int64
chest_pain_type                int64
resting_blood_pressure         int64
cholesterol                    int64
fasting_blood_sugar            int64
resting_electrocardiogram      int64
max_heart_rate_achieved        int64
exercise_induced_angina        int64
st_depression                float64
st_slope                       int64
num_major_vessels              int64
thalassemia                    int64
target                         int64
dtype: object

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC, NuSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.metrics import recall_score, accuracy_score,roc_curve, auc
from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.preprocessing import LabelEncoder

In [16]:
data.head()

,age,sex,chest_pain_type,resting_blood_pressure,cholesterol,fasting_blood_sugar,resting_electrocardiogram,max_heart_rate_achieved,exercise_induced_angina,st_depression,st_slope,num_major_vessels,thalassemia,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1
